# 第6回 分類の判定を、混同行列とROCで評価する


---


## 前回からの接続


第2回から第5回まで、自転車の台数、民泊の料金、住宅の価格と、ずっと数を当ててきました。当たり具合は、平均絶対誤差（MAE）で「何台、何円、何ドル外れるか」と測りました。

今日は、当てたいものが数ではなく、「心臓病か、そうでないか」の2つの区分になります。区分を当てることを、分類（classification）と呼びます。分類では、「何ドル外れたか」という物差しが使えません。当たったか外れたか、外れたならどちらに外れたかを、数えて評価します。


---


## 今日の分析目標


あなたは循環器外来の医師です。検査の値から、心臓病の人をどう見分け、その見分け方をどう評価するかに答える。

冠動脈造影を受けた患者303人のデータで、心臓病かどうかを当てる式を作ります。当たった人と外れた人を4つのマスに数え分け、見逃しと誤検出を分けて評価します。判定の基準（しきい値）を動かすと、何が入れ替わるかも確かめます。

もう1つのねらいは、配列を比べてTrue/Falseを作り、数え上げられるようになることです。混同行列のマスを、ライブラリに頼らず自分で数えます。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、循環器外来の医師です。胸の痛みなどで受診した患者に、冠動脈造影（心臓の血管に造影剤を流して写す検査）を受けてもらうかを決めます。造影は確実に分かる検査ですが、体への負担もあります。

そこで、造影の前に分かる検査の値から、造影で心臓病が見つかりそうな人を見分けたいと考えています。


### 手元のデータ

米国クリーブランド・クリニックで冠動脈造影を受けた患者303人の記録です（Detranoほか, 1989。UCI機械学習リポジトリの心臓病データ）。1行が1人で、年齢・性別・胸痛の種類・血圧・コレステロール・運動したときの心電図などの検査値が13本並びます。

答えの列は、造影の結果、心臓病（心臓の血管が50%以上狭くなっている）と分かったか（1）、そうでないか（0）です。健診の受診者ではなく、心臓の病気を疑われて造影を受けた人たちなので、心臓病の人が半分近くいます。


### 最後に出す答え

検査の値から心臓病を見分けたとき、心臓病の人のうち何人を見逃し、心臓病でない人のうち何人に造影を勧めてしまうかです。そして、判定の基準を動かすと、その2つがどう入れ替わるかです。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 0/1の答えを線形回帰（linear regression）で当て、しきい値で切って判定し、出力が確率と呼べない理由を説明できる
- 正解率だけでは判定を評価できない理由を、「全員を同じ区分と答えた場合」と比べて説明できる
- 混同行列（confusion matrix）の4つのマスを、配列の比較と`&`・`sum()`で自分で数えられる
- 適合率（precision）と再現率（recall）を式から計算し、見逃しと誤検出の違いとして説明できる
- しきい値を動かすと見逃しと誤検出が入れ替わることを表で確かめ、ROC曲線とAUCで判定の力をまとめられる
- 医療の言葉で、感度（sensitivity）と特異度（specificity）を説明できる
- 心臓病の人の割合（有病率）が変わると、感度と特異度が同じでも適合率が変わることを、1,000人あたりの人数で説明できる
- `train_test_split`の`stratify`と`StratifiedKFold`で、区分の割合をそろえて分け、交差検証（cross-validation）で評価の揺れを確かめられる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）

今日は、分類の評価の道具`confusion_matrix`・`roc_curve`・`roc_auc_score`と、割合をそろえた交差検証の道具`StratifiedKFold`が加わります。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### お手本：表を読み込んで、列名を日本語にする

- 役割：表を読み込み、14本の列名を日本語に付け替えて、先頭5行を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名`heart.csv`
- 出力：表`raw`の件数と先頭5行


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/heart.csv')
ja_names = {'age': '年齢', 'sex': '性別', 'cp': '胸痛の種類',
            'trestbps': '安静時血圧', 'chol': 'コレステロール',
            'fbs': '空腹時血糖', 'restecg': '安静時心電図',
            'thalach': '最大心拍数', 'exang': '運動時の胸痛',
            'oldpeak': 'ST低下', 'slope': 'STの傾き',
            'ca': '血管の数', 'thal': 'タリウム検査',
            'target': '心臓病'}
raw = raw.rename(columns=ja_names)
print(len(raw), '人')
raw.head()


各列の中身は次のとおりです。

| 日本語の名前 | 元の名前 | 中身 |
|---|---|---|
| `年齢`・`性別` | `age`・`sex` | 年齢と性別（1が男性、0が女性） |
| `胸痛の種類` | `cp` | 1：典型的な狭心症、2：非典型的な狭心症、3：狭心症ではない痛み、4：痛みなし |
| `安静時血圧`・`コレステロール` | `trestbps`・`chol` | 安静にしたときの血圧（mmHg）と、血中コレステロール（mg/dl） |
| `空腹時血糖` | `fbs` | 空腹時の血糖値が120 mg/dlを超えるか（1）、そうでないか（0） |
| `安静時心電図` | `restecg` | 0：正常、1：STやTの波形の異常、2：左心室の肥大 |
| `最大心拍数` | `thalach` | 運動の検査で達した、いちばん高い心拍数 |
| `運動時の胸痛` | `exang` | 運動の検査で胸が痛くなったか（1）、そうでないか（0） |
| `ST低下`・`STの傾き` | `oldpeak`・`slope` | 運動したときの心電図のST部分の下がり方と、その傾き（1：上がり、2：平ら、3：下がり） |
| `血管の数` | `ca` | 透視で写った太い血管の数（0〜3） |
| `タリウム検査` | `thal` | 心臓の血の流れの検査（3：正常、6：固定した欠損、7：回復する欠損） |
| `心臓病` | `target` | 造影で心臓病と分かったか（1）、そうでないか（0）。当てたい値 |


### 変えてみる：欠測を確かめる

- 役割：列ごとの欠測（missing data）の数と、心臓病の人の割合を出す
- 入力：表`raw`
- 出力：欠測のある列と件数、心臓病の人の割合

`raw['心臓病'].mean()`は、1と0の並びの平均なので、1（心臓病）の割合になります。


In [ ]:
print(raw.isna().sum()[raw.isna().sum() > 0])
print(round(raw['心臓病'].mean(), 3))


欠測しているのは、血管の数が4人、タリウム検査が2人の、合わせて6人です。欠けた理由は、データからは分かりません。心臓病の人の割合は0.459で、半分近くです。

第5回で見たように、欠測は、真の値に戻す・予測して補完（imputation）する・使わない、の3つのなくし方があります。今回は、理由が分からず、303人のうち6人と少ないので、この6人の行を使わないことにします。


### お手本：欠測の行を外し、区分の列をダミー変数にする

- 役割：欠測のある6人を外し、番号で区分を表す4本の列をダミー変数（dummy variable）にして、手がかりの表`X`と答え`y`を作る
- 入力：表`raw`
- 出力：人数と、手がかりの表の大きさ

`dropna()`は、欠測のある行を外すメソッドです。胸痛の種類・安静時心電図・STの傾き・タリウム検査は、数で書いてありますが区分です。第3回で見たように、番号のまま式に入れると「番号が1つ大きいと」という意味のない係数になるので、`get_dummies`でダミー変数にします。どの区分があるかは検査の決まりで決まっているので、データ全体で作ってかまいません。


In [ ]:
df = raw.dropna().reset_index(drop=True)
print(len(df), '人')
kinds = ['胸痛の種類', '安静時心電図', 'STの傾き', 'タリウム検査']
X = pd.get_dummies(df.drop(columns='心臓病'), columns=kinds,
                   drop_first=True, dtype=int)
y = df['心臓病']
print(X.shape)


297人になりました。手がかりは、13本の検査値のうち4本がダミー変数に分かれて、18本です。


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、②の解析（分類）と、その評価が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i == 2
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の8つの小さな問いに答えていきます。

1. どんな人たちか？（1節）
2. 0/1を線形回帰で当てると？（2節）
3. どれだけ当たった？（3節）
4. 見逃しと誤検出は？（4節）
5. 心臓病の人の割合が変わると？（5節）
6. しきい値を動かすと？（6節）
7. その数は、分け方で揺れないか？（7節）
8. 医師として、何と言えるか？（8節）


---


## 1. どんな人たちか？


手がかりの表ができました。式を作る前に、心臓病の人とそうでない人で、検査の値がどう違うかを見ておきます。


### お手本：2つのグループの平均を比べる

- 役割：心臓病の人とそうでない人に分けて、人数と、5つの検査値の平均を出す
- 入力：表`df`
- 出力：グループごとの人数と平均

`groupby('心臓病')`で、心臓病の列の値（0と1）ごとに分けて集計します（第3回のおさらい）。


In [ ]:
print(df['心臓病'].value_counts())
nums = ['年齢', '最大心拍数', 'ST低下', '安静時血圧',
        'コレステロール']
print(df.groupby('心臓病')[nums].mean().round(1))


心臓病の人（1）は137人、そうでない人（0）は160人です。心臓病の人は、平均で、年齢が4歳ほど高く（56.8歳と52.6歳）、運動の検査で達した最大心拍数が低く（139.1と158.6）、ST低下が大きい（1.6と0.6）です。血圧やコレステロールの差は、それに比べて小さめです。

ただ、平均が違っても、1人ずつ見ると重なっています。1つの検査値だけでは見分けきれないので、18本の手がかりをまとめて使います。


In [ ]:
plt.hist(df[df['心臓病'] == 0]['最大心拍数'], bins=20, alpha=0.6,
         label='心臓病でない')
plt.hist(df[df['心臓病'] == 1]['最大心拍数'], bins=20, alpha=0.6,
         label='心臓病')
plt.xlabel('最大心拍数')
plt.ylabel('人数')
plt.legend()
plt.show()


最大心拍数の分布は、2つのグループで山の位置がずれていますが、大きく重なっています。


### お手本：割合をそろえて、訓練とテストに分ける

- 役割：297人を訓練8割・テスト2割に分ける。心臓病の人の割合を、訓練とテストでそろえる
- 入力：手がかりの表`X`と、答え`y`
- 出力：訓練とテストの人数と、それぞれの心臓病の人の割合、テストの心臓病の人数

第2回と同じ`train_test_split`です。今日は`stratify=y`を足します。`stratify`は「層に分ける」という意味で、`y`の区分（0と1）の割合が、訓練とテストで同じになるように分けます。テストが60人と少ないので、たまたま心臓病の人が偏らないようにするためです。


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
print(len(X_tr), len(X_te))
print(round(y_tr.mean(), 3), round(y_te.mean(), 3))
print(y_te.sum(), '人がテストの心臓病')


訓練237人、テスト60人です。心臓病の人の割合は、訓練で0.46、テストで0.467と、ほぼそろいました。テストの60人のうち、心臓病の人は28人、そうでない人は32人です。


---


## 2. 0/1を線形回帰で当てると？


訓練とテストに分けました。区分を当てる道具は、まだ持っていません。でも、心臓病を1、そうでないを0という数だと思えば、これまでの線形回帰で当てられそうです。


### お手本：心臓病（0/1）を線形回帰で当てる

- 役割：訓練の人で、18本の手がかりから心臓病（0/1）を当てる線形回帰の式を作り、テストの人の値を出す
- 入力：訓練の`X_tr`・`y_tr`と、テストの`X_te`
- 出力：テストの人の、式の値の最小と最大、0〜1をはみ出した人数

式の値を$s_i$と書きます。$s_i$が1に近いほど心臓病らしく、0に近いほど心臓病でなさそう、と読めます。


In [ ]:
m = LinearRegression().fit(X_tr, y_tr)
s = m.predict(X_te)                 # テストの人の、式の値
print(round(s.min(), 2), round(s.max(), 2))
print(((s < 0) | (s > 1)).sum(), '人が0〜1の外')


### お手本：しきい値で切って、判定にする

式の値は、0.33や0.86のような小数です。「心臓病」か「そうでない」かに決めるには、どこかで切ります。この切れ目を、しきい値（threshold）と呼びます。まずは、0と1のまん中の0.5で切ります。

$$
\hat{y}_i = \begin{cases} 1 & (s_i \ge 0.5) \\ 0 & (s_i < 0.5) \end{cases}
$$

- 役割：式の値を0.5で切って、テストの人を心臓病（1）とそうでない（0）に判定する
- 入力：式の値`s`
- 出力：先頭5人の、式の値と判定と答え

`s >= 0.5`は、並びの1つずつを0.5と比べて、TrueかFalseの並びを返します（第1回の条件で絞るときと同じ比べ方です）。`.astype(int)`は、Trueを1、Falseを0の数に変えます。


In [ ]:
pred = (s >= 0.5).astype(int)       # 0.5以上なら心臓病（1）
print(s[:5].round(2))
print(pred[:5])
print(y_te.values[:5])


先頭の5人では、1人目（0.33）が0に、ほかの4人が1に判定されました。いちばん下の答えと見比べると、2〜5人目は当たっています。ところが1人目は、答えが1（心臓病）なのに、0（心臓病でない）と判定されています。この外れ方は、あとで詳しく見ます。


### 問い：式の値は、心臓病の確率か？

式の値は、テストの人で最小−0.21、最大1.17でした。60人のうち7人が、0〜1の外にいます。この値を「心臓病である確率」と呼んでよいでしょうか。考えを下に書いてください。


（ここに考えを書く）


呼べません。確率は0から1の間の数ですが、線形回帰は直線（平面）なので、手がかりの値が大きければ1を超え、小さければ0を下回ります。−0.21の人は「心臓病である確率がマイナス」になってしまいます。

それでも、式の値の大小には意味があります。値が大きい人ほど、心臓病の人に似た検査値を持っています。並べる順番と、どこかで切った判定には使えます。0〜1に収まる確率を出す方法は、今日は扱いません。今日は、この判定がどれだけ当たっているかを評価します。


---


## 3. どれだけ当たった？


判定ができました。テストの60人のうち、何人が当たったのでしょうか。まずは、当たった人の割合から測ります。


### お手本：正解率を出す

- 役割：判定と答えを1人ずつ比べ、当たった人の割合（正解率）を出す
- 入力：判定`pred`と答え`y_te`
- 出力：当たった人数と正解率、全員を「心臓病でない」と答えたときの正解率

`pred == y_te`は、1人ずつ判定と答えが同じか（True）違うか（False）の並びです。Trueは1、Falseは0として数えるので、`.mean()`で当たった人の割合になります（第3回の`isna().mean()`と同じ考え方です）。

正解率（accuracy）は、次の式です。$n$はテストの人数、$[\hat{y}_i = y_i]$は当たったら1、外れたら0です。

$$
\text{正解率} = \frac{1}{n}\sum_{i=1}^{n} [\hat{y}_i = y_i]
$$


In [ ]:
print((pred == y_te).sum(), '人が当たり')
print(round((pred == y_te).mean(), 3))
print(round((y_te == 0).mean(), 3))   # 全員を「でない」と答えたら


正解率は0.85です。60人のうち51人が当たりました。

ただ、何も考えずに全員を「心臓病でない」と答えても、正解率は0.533になります。テストの60人のうち32人は心臓病でないからです。しかも、その答え方では、心臓病の28人を全員見逃します。正解率は、どちらに外れたかを区別しません。外れた9人の中身を、分けて数える必要があります。


### 判定と答えの組は、4通り

1人1人の判定と答えの組は、次の4通りのどれかです。心臓病を「陽性」、心臓病でないを「陰性」と呼びます。

| | 判定：陰性（0） | 判定：陽性（1） |
|---|---|---|
| 答え：陰性（0） | 真陰性（TN）：正しく「心臓病でない」 | 偽陽性（FP）：誤検出。健康なのに造影を勧める |
| 答え：陽性（1） | 偽陰性（FN）：見逃し。心臓病なのに見逃す | 真陽性（TP）：正しく「心臓病」 |

TPはtrue positive、TNはtrue negative、FPはfalse positive、FNはfalse negativeの略です。この4つのマスの表を、混同行列（confusion matrix）と呼びます。


### お手本：真陽性（TP）を数える

- 役割：判定が1で、答えも1の人（真陽性）を数える
- 入力：判定`pred`と答え`y_te`
- 出力：真陽性の人数

`(pred == 1)`と`(y_te == 1)`は、どちらもTrue/Falseの並びです。`&`は「かつ」で、1人ずつ両方がTrueのときだけTrueにします（第3回のおさらい）。`.sum()`で、Trueの数を数えます。

`y_te`はpandasの列なので、`.values`で、行の番号の付かない数の並びにしてから比べます。`pred`と同じ順に並んだ、ただの並びどうしの比較になります。


In [ ]:
y_true = y_te.values                 # 答えを、ただの並びにする
tp = ((pred == 1) & (y_true == 1)).sum()
print(tp, '人が真陽性')


真陽性は22人です。テストの心臓病の28人のうち、22人を正しく「心臓病」と判定しました。


### 変えてみる：真陰性（TN）を数える

- 役割：判定が0で、答えも0の人（真陰性）を数える
- 入力：判定`pred`と答え`y_true`
- 出力：真陰性の人数

お手本の`== 1`を、2か所とも`== 0`に変えます。


In [ ]:
tn = ((pred == 0) & (y_true == 0)).sum()
print(tn, '人が真陰性')


真陰性は29人です。心臓病でない32人のうち、29人を正しく「心臓病でない」と判定しました。真陽性の22人と真陰性の29人を足すと51人で、正解率0.85の51人と同じです。


### TODO①：誤検出（FP）と見逃し（FN）を数える

お手本と同じ形で、偽陽性（判定が1で答えが0）の人数`fp`と、偽陰性（判定が0で答えが1）の人数`fn`を数えて表示してください。そのあと、`confusion_matrix(y_true, pred)`で、scikit-learnの混同行列と比べてください。


In [ ]:
# TODO: fp と fn を数えて表示し、confusion_matrix と比べる
# ヒント: fp = ((pred == 1) & (y_true == 0)).sum()
#         fn は、判定が 0 で答えが 1
...


偽陽性（誤検出）は3人、偽陰性（見逃し）は6人です。4つのマスを足すと、22 + 29 + 3 + 6 = 60人で、テストの全員になります。

`confusion_matrix`は、`[[TN, FP], [FN, TP]]`の形の表を返します。1行目が答え0の人、2行目が答え1の人、1列目が判定0、2列目が判定1です。自分で数えた4つの数と同じ、`[[29, 3], [6, 22]]`になります。


In [ ]:
# @title 図：混同行列
cm = np.array([[((pred == 0) & (y_true == 0)).sum(), ((pred == 1) & (y_true == 0)).sum()],
               [((pred == 0) & (y_true == 1)).sum(), ((pred == 1) & (y_true == 1)).sum()]])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Blues')
names = [['真陰性 TN', '偽陽性 FP\n（誤検出）'], ['偽陰性 FN\n（見逃し）', '真陽性 TP']]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{names[i][j]}\n{cm[i, j]}人', ha='center', va='center',
                color='white' if cm[i, j] > 15 else 'black', fontsize=12)
ax.set_xticks([0, 1]); ax.set_xticklabels(['判定：陰性', '判定：陽性'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['答え：陰性', '答え：陽性'])
plt.show()


---


## 4. 見逃しと誤検出は？


4つのマスが数えられました。外れた9人は、見逃しが6人、誤検出が3人です。この2つは、医師にとって重さが違います。見逃せば、心臓病の人が治療を受けられません。誤検出なら、健康な人が負担のある造影を受けます。それぞれを、割合の物差しにします。


### 2つの物差し：適合率と再現率

適合率（precision）は、「心臓病」と判定した人のうち、本当に心臓病だった人の割合です。誤検出が少ないほど大きくなります。

$$
\text{適合率} = \frac{\text{TP}}{\text{TP} + \text{FP}}
$$

再現率（recall）は、本当に心臓病の人のうち、「心臓病」と判定できた人の割合です。見逃しが少ないほど大きくなります。

$$
\text{再現率} = \frac{\text{TP}}{\text{TP} + \text{FN}}
$$

分母が違います。適合率の分母は「判定が陽性の人」、再現率の分母は「答えが陽性の人」です。


### TODO②：適合率と再現率を、式から計算する

3節で数えた`tp`と、TODO①で数えた`fp`・`fn`を使って、適合率と再現率を計算し、小数第3位まで表示してください。

TODO①を飛ばした人のために、`fp`と`fn`の数え方もヒントに書いてあります。


In [ ]:
# TODO: tp, fp, fn から適合率と再現率を計算し、小数第3位まで表示する
# ヒント: fp と fn を先に数える（TODO①と同じ）。
#         適合率は tp / (tp + fp)、再現率は tp / (tp + fn)
...


適合率は0.88、再現率は0.786です。「心臓病」と判定した25人のうち22人は本当に心臓病で、誤検出は3人でした。一方、本当に心臓病の28人のうち、見つけられたのは22人で、6人を見逃しています。

医師としては、見逃しの6人が気になります。心臓病の人の5人に1人ほどを、造影に回さずに帰してしまうことになるからです。正解率0.85という1つの数では、この6人は見えませんでした。


### 医療の言葉：感度と特異度

医療の検査では、同じ物差しを別の名前で呼びます。

- 感度（sensitivity）：本当に病気の人のうち、陽性と判定できた人の割合。再現率と同じです
- 特異度（specificity）：本当は病気でない人のうち、陰性と判定できた人の割合。$\text{TN} / (\text{TN} + \text{FP})$です

感度が高い検査は、見逃しが少ない検査です。特異度が高い検査は、健康な人を正しく「病気でない」と言える、誤検出の少ない検査です。

- 役割：感度と特異度を出す
- 入力：判定`pred`、答え`y_true`、3節の`tp`・`tn`
- 出力：感度と特異度


In [ ]:
fp = ((pred == 1) & (y_true == 0)).sum()
fn = ((pred == 0) & (y_true == 1)).sum()
print(round(tp / (tp + fn), 3))   # 感度（再現率）
print(round(tn / (tn + fp), 3))   # 特異度


感度は0.786、特異度は0.906です。心臓病でない32人のうち29人を、正しく「心臓病でない」と判定しています。

感度と特異度は、どちらも「答えが同じ人たちの中での割合」です。感度は心臓病の人の中、特異度は心臓病でない人の中で数えます。この性質が、次の節で効いてきます。


---


## 5. 心臓病の人の割合が変わると？


感度0.786、特異度0.906の判定の式ができました。この式を、外来ではなく、健診のような場に持ち込んだら、どうなるでしょうか。導入で見たとおり、このデータは心臓病を疑われて造影を受けた人たちで、心臓病の人が46%もいます。健診の受診者なら、心臓病の人はずっと少ないはずです。


### 有病率

ある集団の中で、病気の人の割合を、有病率（prevalence）と呼びます。このデータの有病率は約0.46です。健診の受診者の有病率は、例えば0.05（20人に1人）のように、ずっと小さくなります。

感度と特異度は、病気の人の中・病気でない人の中で数えた割合なので、有病率が変わっても、同じ検査なら変わりません。では、適合率はどうでしょうか。適合率の分母は「陽性と判定した人」で、病気の人と病気でない人の両方が混ざっています。


### お手本：1,000人あたりで数える

割合のままだと分かりにくいので、1,000人の集団を考えて、人数で数えます。有病率が$r$のとき、1,000人のうち心臓病の人は$1000r$人、そうでない人は$1000(1-r)$人です。

- 正しく見つかる人（真陽性）：心臓病の人 × 感度
- 誤検出（偽陽性）：心臓病でない人 × 誤検出率（= 1 − 特異度）

- 役割：有病率を0.46・0.2・0.05に変えて、1,000人あたりの真陽性・誤検出と、適合率を表にする
- 入力：テストで測った感度`sens`と誤検出率`fpr_05`
- 出力：有病率ごとの人数と適合率の表


In [ ]:
sens = tp / (tp + fn)            # 感度（再現率）
fpr_05 = fp / (fp + tn)          # 誤検出率（1 − 特異度）
rows = []
for r in [0.46, 0.2, 0.05]:
    n_sick = 1000 * r                    # 1,000人のうち心臓病の人
    found = n_sick * sens                # 正しく見つかる人
    wrong = (1000 - n_sick) * fpr_05     # 誤検出
    rows.append({'有病率': r, '心臓病': round(n_sick),
                 '見つかる': round(found), '誤検出': round(wrong),
                 '適合率': round(found / (found + wrong), 3)})
print(pd.DataFrame(rows))


有病率0.46（このデータ）では、1,000人のうち心臓病の460人から361人が見つかり、誤検出は51人です。適合率は0.877で、テストの0.88とほぼ同じです。

有病率を0.05にすると、心臓病の人は50人に減り、見つかるのは39人です。ところが、心臓病でない人は950人に増えるので、誤検出は89人になります。適合率は0.306で、「心臓病」と判定した人の7割が健康な人です。

感度も特異度も同じ式なのに、使う場の有病率が下がると、陽性の判定の多くが誤検出になります。外来で作った判定の式を、そのまま健診に持ち込むことはできません。


In [ ]:
# @title 図：1,000人で見る（有病率0.46と0.05）
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, r in zip(axes, [0.46, 0.05]):
    n_sick = round(1000 * r)
    found = round(n_sick * sens)
    wrong = round((1000 - n_sick) * fpr_05)
    kinds = (['見つかる'] * found + ['見逃し'] * (n_sick - found)
             + ['誤検出'] * wrong + ['正しく陰性'] * (1000 - n_sick - wrong))
    col = {'見つかる': '#e63946', '見逃し': '#f4a261',
           '誤検出': '#0066cc', '正しく陰性': '#dddddd'}
    for k, kind in enumerate(kinds):
        ax.scatter(k % 40, -(k // 40), s=9, color=col[kind])
    ax.set_title(f'有病率{r}：見つかる{found}人・誤検出{wrong}人')
    ax.axis('off')
plt.show()


赤が正しく見つかる人、橙が見逃し、青が誤検出、灰色が正しく「でない」と判定された人です。有病率0.05の右の図では、陽性と判定された人（赤と青）のうち、青のほうがずっと多くなります。

医療では、適合率を陽性的中率（positive predictive value）とも呼びます。陽性と判定された人が、本当に病気である割合です。陽性的中率は、検査の性能（感度と特異度）だけでなく、使う場の有病率で決まります。


### 変えてみる：有病率0.01なら

- 役割：有病率を0.01（100人に1人）にして、同じ計算をする
- 入力：お手本の`for`の中身を、`r = 0.01`の1回分だけ書く
- 出力：1,000人あたりの人数と適合率


In [ ]:
r = 0.01
n_sick = 1000 * r
found = n_sick * sens
wrong = (1000 - n_sick) * fpr_05
print(round(n_sick), round(found), round(wrong),
      round(found / (found + wrong), 3))


有病率0.01では、1,000人のうち心臓病の10人から8人が見つかりますが、誤検出は93人です。適合率は0.078で、陽性と判定した人の9割以上が健康な人になります。


---


## 6. しきい値を動かすと？


見逃しを減らすには、どうすればよいでしょうか。ここまでは、しきい値を0.5に決めて切りました。しきい値を下げれば、式の値が小さめの人も「心臓病」と判定されるので、見逃しは減りそうです。そのかわり、誤検出は増えそうです。


### お手本：しきい値ごとに、4つのマスを数える

- 役割：しきい値を0.1から0.9まで変えて、それぞれで判定し、TP・FP・FN・TNを数えた表を作る
- 入力：式の値`s`と答え`y_true`
- 出力：しきい値ごとの4つのマスの人数の表

3節の数え方を、`for`文の中でくり返します。`rows.append`で1行ずつ辞書をため、最後に表にするのは、第4回と同じ形です。`np.arange(0.1, 1.0, 0.1)`は、0.1から1.0の手前まで0.1刻みの並びで、`round(t, 1)`で0.1・0.2のようにきれいな数にします。


In [ ]:
rows = []
for t in np.arange(0.1, 1.0, 0.1):
    t = round(t, 1)
    p = (s >= t).astype(int)                    # しきい値 t で判定
    rows.append({'しきい値': t,
                 'TP': ((p == 1) & (y_true == 1)).sum(),
                 'FP': ((p == 1) & (y_true == 0)).sum(),
                 'FN': ((p == 0) & (y_true == 1)).sum(),
                 'TN': ((p == 0) & (y_true == 0)).sum()})
table = pd.DataFrame(rows)
print(table)


しきい値を0.1まで下げると、見逃し（FN）は0人になります。心臓病の28人を全員見つけられます。そのかわり、誤検出（FP）は17人に増え、心臓病でない32人の半分以上に造影を勧めることになります。

反対に、しきい値を0.9まで上げると、誤検出は0人になりますが、見逃しは20人に増えます。

しきい値を動かすと、見逃しと誤検出が入れ替わるのです。どちらも0にする切り方は、ありません。


### TODO③：再現率と誤検出率の列を足す

お手本の表`table`に、2つの列を足してください。1つは再現率（`TP / (TP + FN)`）、もう1つは誤検出率です。誤検出率（false positive rate）は、本当は心臓病でない人のうち、誤って「心臓病」と判定した人の割合です。

$$
\text{誤検出率} = \frac{\text{FP}}{\text{FP} + \text{TN}}
$$

表の列どうしの計算は、`table['TP'] / (table['TP'] + table['FN'])`のように書けます。


In [ ]:
# TODO: table に '再現率' と '誤検出率' の列を足して表示する
# ヒント: table['再現率'] = table['TP'] / (table['TP'] + table['FN'])
#         誤検出率は FP / (FP + TN)
...


しきい値を0.1から0.9へ上げていくと、再現率は1.0から0.286へ、誤検出率は0.531から0へ、どちらも下がっていきます。再現率は高いほど、誤検出率は低いほど良いので、2つを同時に良くすることはできません。


### ROC曲線：すべてのしきい値を1枚に

しきい値を細かく動かして、横に誤検出率、縦に再現率をとって点を結んだ曲線を、ROC曲線（receiver operating characteristic curve）と呼びます。再現率は、ROC曲線では真陽性率（true positive rate）とも呼びます。

左上の角（誤検出率0、再現率1）に近いほど、誤検出を増やさずに心臓病の人を見つけられる、良い判定の式です。式の値がでたらめなら、曲線は左下から右上への対角線になります。

- 役割：ROC曲線を描き、曲線の下の面積（AUC）を出す
- 入力：答え`y_true`と、式の値`s`
- 出力：ROC曲線の図と、AUC

`roc_curve(答え, 値)`は、しきい値を細かく動かしたときの誤検出率`fpr`と真陽性率`tpr`の並びを返します。`roc_auc_score(答え, 値)`は、曲線の下の面積を返します。どちらも、判定（0/1）ではなく、切る前の値`s`を渡します。


In [ ]:
fpr, tpr, thr = roc_curve(y_true, s)
plt.plot(fpr, tpr, label='線形回帰の式')
plt.plot([0, 1], [0, 1], '--', color='gray', label='でたらめ')
plt.xlabel('誤検出率（FP / (FP + TN)）')
plt.ylabel('再現率（TP / (TP + FN)）')
plt.legend()
plt.show()
print(round(roc_auc_score(y_true, s), 3))


ROC曲線は、対角線よりずっと左上に張り出しています。曲線の下の面積を、AUC（area under the curve）と呼びます。でたらめなら0.5、完璧に見分けられれば1.0です。この式のAUCは0.929です。

AUCは、しきい値を1つに決めずに、式の値がどれだけ心臓病の人とそうでない人を分けて並べられるかを表します。しきい値をどこにするかは、見逃しと誤検出のどちらをどれだけ重く見るかで決まります。今日は決めずに、次回に回します。


---


## 7. その数は、分け方で揺れないか？


ここまでの数は、すべてテストの60人で測りました。60人は少ない人数です。第4回で見たように、1回の分け方の数は、分け方しだいで揺れます。分類の評価でも、交差検証で確かめます。


### 割合をそろえた交差検証

第4回の`KFold`は、行を混ぜて5つの組に分けました。分類では、組ごとに心臓病の人の割合が偏らないよう、`StratifiedKFold`を使います。`stratify`と同じく、区分の割合を、どの組でもそろえて分けます。

`skf.split(X, y)`のように、`KFold`と違って答え`y`も渡します。割合をそろえるために、どの行が心臓病かを知る必要があるからです。

- 役割：297人を、心臓病の割合をそろえた5つの組に分け、各回で式を作り、見逃し・誤検出・感度・AUCをためた表を作る
- 入力：手がかりの表`X`と答え`y`
- 出力：5回分の表と、その平均

各回の中身は、2節（式を作って0.5で切る）と3節（4つのマスを数える）のセルを並べたものです。


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
rows = []
for tr_i, te_i in skf.split(X, y):
    m_s = LinearRegression().fit(X.iloc[tr_i], y.iloc[tr_i])
    s_s = m_s.predict(X.iloc[te_i])        # その回のテストの値
    y_s = y.iloc[te_i].values
    p_s = (s_s >= 0.5).astype(int)
    tp_s = ((p_s == 1) & (y_s == 1)).sum()
    fn_s = ((p_s == 0) & (y_s == 1)).sum()
    rows.append({'見逃し': fn_s,
                 '誤検出': ((p_s == 1) & (y_s == 0)).sum(),
                 '感度': round(tp_s / (tp_s + fn_s), 3),
                 'AUC': round(roc_auc_score(y_s, s_s), 3)})
cv = pd.DataFrame(rows)
print(cv)
print(cv.mean().round(3))


5回のテストで、見逃しは3人から8人まで、感度は0.704から0.893まで揺れます。AUCも0.863から0.952まで揺れます。平均は、感度0.78、AUC 0.906です。

2節からのテスト（感度0.786、AUC 0.929）は、この5回の範囲の中の1回にあたります。「見逃し6人」は、たまたまその分け方での人数です。報告するときは、交差検証の平均と、その揺れの幅も一緒に伝えます。


In [ ]:
plt.plot(range(1, 6), cv['感度'], 'o-', label='感度')
plt.plot(range(1, 6), cv['AUC'], 's-', label='AUC')
plt.xlabel('交差検証の回')
plt.ylabel('値')
plt.ylim(0.6, 1.0)
plt.legend()
plt.show()


### TODO④：特異度の列も足す

交差検証のセルを写し、各回の特異度（`TN / (TN + FP)`）も表に加えてください。特異度の5回の範囲と平均を確かめましょう。


In [ ]:
# TODO: 交差検証の各回で特異度を計算し、表に加える
# ヒント: tn_s と fp_s も数え、rows.append の辞書に
#         '特異度': round(tn_s / (tn_s + fp_s), 3) を足す
...


特異度は、5回で0.875から0.969まで揺れ、平均は0.9です。誤検出は1人から4人です。


---


## 8. 医師として、何と言えるか？


評価の物差しがそろいました。最後に、医師として、この判定の式について何と言えるかを、数でまとめます。


### お手本：見逃しを文にする

- 役割：しきい値0.5のときの見逃しの人数を、心臓病の人数とあわせて文にする
- 入力：判定`pred`と答え`y_true`
- 出力：見逃しを説明する文

f-stringで、数え上げた値をそのまま文に入れます（第4回のおさらい）。`{rec:.1%}`の`.1%`は、割合を百分率（小数第1位）で書く指定です。


In [ ]:
fn = ((pred == 0) & (y_true == 1)).sum()
n_sick = (y_true == 1).sum()
rec = (n_sick - fn) / n_sick
print(f'心臓病の{n_sick}人のうち{fn}人を見逃し、'
      f'再現率は{rec:.1%}です。')


「心臓病の28人のうち6人を見逃し、再現率は78.6%です。」という文になりました。


### TODO⑤：誤検出も文にする

お手本をまねて、心臓病でない人の人数`n_ok`と、そのうち誤って「心臓病」と判定した人数`fp`を数え、「心臓病でない○人のうち○人に造影を勧め、誤検出率は○%です。」という文を表示してください。


In [ ]:
# TODO: 心臓病でない人数 n_ok と誤検出 fp を数え、文にする
# ヒント: n_ok = (y_true == 0).sum()、fp は TODO①と同じ数え方
#         お手本の f-string をまねる
...


「心臓病でない32人のうち3人に造影を勧め、誤検出率は9.4%です。」となります。


### 答え

18本の検査値から線形回帰で心臓病（0/1）を当て、0.5で切って判定すると、テストの60人での正解率は0.85でした。ただし、正解率だけでは足りません。心臓病の28人のうち6人を見逃し（再現率78.6%）、心臓病でない32人のうち3人に造影を勧めます（誤検出率9.4%）。

しきい値を下げれば見逃しは減り、0.1では見逃し0人になりますが、誤検出は17人に増えます。どの切り方でも、見逃しと誤検出の両方を0にはできません。しきい値の切り方によらない判定の力は、AUC 0.929です。

この数は、分け方で揺れます。5分割の交差検証では、感度は0.704〜0.893（平均0.78）、AUCは0.863〜0.952（平均0.906）でした。そして、この式は外来の患者で作ったものです。有病率の低い健診に持ち込むと、陽性と判定した人の多くが誤検出になります（有病率0.05で適合率0.306）。


まだ言えないこと

- しきい値をどこに決めるか：見逃しと誤検出の重さをどう比べるかで決まる
- 確率：線形回帰の式の値は0〜1をはみ出し、確率とは呼べない。0〜1に収まる確率を出すには、別の式が要る
- ほかの病院で使えるか：このデータは1つの病院の、造影を受けた患者だけ。患者の集まり方が違う病院では、感度や特異度も変わりうる


---


## 目標に答えられたか


- 今日の目標は「検査の値から、心臓病の人をどう見分け、その見分け方をどう評価するか」でした。答えを、見逃しと誤検出の人数で説明できますか？
- 2節で、線形回帰の式の値を「心臓病の確率」と呼べなかったのはなぜですか？
- 3節で、正解率0.85だけでは評価が足りないのはなぜですか？ 全員を「心臓病でない」と答えたら、正解率はいくつでしたか？
- 3節の混同行列の4つのマスは、それぞれ何を表していましたか？ 自分のコードで、どう数えましたか？
- 4節の適合率と再現率は、分母が何でしたか？ 医師にとって、どちらの外れが重いでしょうか？
- 4節の感度と特異度は、それぞれどの人たちの中で数えた割合ですか？
- 5節で、有病率を0.05にすると、適合率はどうなりましたか？ なぜでしょうか？
- 6節で、しきい値を下げると、見逃しと誤検出はどう変わりましたか？ AUCは、何を表す数でしたか？
- 7節の交差検証で、感度はどれくらい揺れましたか？


---


## 今日の要点


- 今日、持ち帰る1文：判定の良し悪しは正解率だけで語らない。混同行列の4つのマスを数え、見逃しと誤検出を分けて評価する。しきい値を動かすと、見逃しと誤検出は入れ替わる
- 0/1の答えを線形回帰で当て、しきい値で切れば判定できる。ただし式の値は0〜1をはみ出し、確率とは呼べない
- 正解率は、全員を同じ区分と答えても高くなることがある。どちらに外れたかを区別しない
- 適合率は判定が陽性の人のうちの当たり（誤検出の少なさ）、再現率は答えが陽性の人のうちの当たり（見逃しの少なさ）
- 医療では、再現率を感度、陰性の人のうちの当たりを特異度と呼ぶ。どちらも有病率によらない
- 適合率（陽性的中率）は、使う場の有病率で変わる。有病率が低いと、陽性の判定の多くが誤検出になる
- ROC曲線は、すべてのしきい値での再現率と誤検出率を1枚に描く。AUCは、しきい値によらない判定の力
- 評価の数も分け方で揺れる。割合をそろえた交差検証（`StratifiedKFold`）で、平均と揺れの幅を見る
- Python：配列を比べるとTrue/Falseの並びになる。`&`で「かつ」、`sum()`で数、`mean()`で割合。同じ数え方を`for`でくり返して表にする


---


## 次回へ


今日は、線形回帰の式の値を0.5で切って判定しました。でも、その値は確率ではなく、しきい値をどこに決めればよいかも分かっていません。

次回の問いは、0〜1に収まる「心臓病である確率」を出し、どの確率から造影を勧めるかを、どう決めるか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

1節で平均の差が大きかった、年齢・最大心拍数・ST低下の3本の列だけで、同じ評価をしてください。訓練の人でこの3本だけの線形回帰の式を作り、テストの人の式の値を0.5で切って判定し、TP・FP・FNと再現率、そして式の値のAUCを表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

3本の列の名前を`cols3 = ['年齢', '最大心拍数', 'ST低下']`にしまい、`X_tr[cols3]`で式を作ります。
テストの値は`X_te[cols3]`で出します。判定と数え方は、2節と3節のセルと同じです。AUCは`roc_auc_score(y_true, 式の値)`です。

</details>


### 応用②（判断）

応用①の3本の列の式で、0.5で切ったときの再現率はいくつですか。小数第3位まで答えてください（例: 0.123）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

18本の式の再現率は0.786でした。

</details>


### 応用③（解釈）

応用①の結果と、本編の18本の式を比べて、外来でどちらの式を使うことを勧めるかを、見逃しと誤検出の数、AUCを使って、同僚の医師に向けて3行程度で書いてください。


（ここに3行程度で書く）


---


## 発展（任意）


### AUCを、組を数えて確かめる

5節のAUCは、ROC曲線の下の面積でした。実は、AUCには、もう1つの読み方があります。テストの心臓病の人を1人、心臓病でない人を1人選んだ組のうち、心臓病の人の式の値のほうが大きい組の割合です。

心臓病の28人と心臓病でない32人なら、組は28 × 32 = 896組あります。`for`文を2重にして、すべての組を比べて数えます。

- 役割：心臓病の人と心臓病でない人のすべての組で、式の値を比べ、心臓病の人のほうが大きい組を数える
- 入力：式の値`s`と答え`y_true`
- 出力：組の数、心臓病の人のほうが大きい組の数、その割合

`s[y_true == 1]`は、答えが1の人の式の値だけを取り出します（条件で絞る書き方です）。


In [ ]:
sick = s[y_true == 1]       # 心臓病の人の式の値
ok = s[y_true == 0]         # 心臓病でない人の式の値
count = 0
for a in sick:
    for b in ok:
        if a > b:
            count += 1
pairs = len(sick) * len(ok)            # 組の数
print(pairs, count, round(count / pairs, 3))


896組のうち832組で、心臓病の人の式の値のほうが大きくなりました。割合は0.929で、`roc_auc_score`のAUCと同じです。

AUCは、「心臓病の人とそうでない人を1人ずつ選んだとき、式が心臓病の人を正しく上に並べる確率」です。しきい値をどこに決めるかとは関係なく、式の値の並べ方のよさを表します。


### 線形回帰の係数の向きは、線形判別分析と同じ

0/1を線形回帰で当てた式の係数は、もう1つの分類の方法である線形判別分析（LDA, linear discriminant analysis）の係数と、向きが同じになることが知られています。scikit-learnの`LinearDiscriminantAnalysis`で確かめます。初めての道具なので、`help(LinearDiscriminantAnalysis)`で、属性（Attributes）の`coef_`の説明を読んでから使ってみましょう。

2つの係数の並び（18個の数）を、18次元の矢印と見て、向きがどれだけそろっているかを、コサイン類似度で測ります。1なら同じ向き、0なら直角です。

$$
\cos\theta = \frac{\boldsymbol{a} \cdot \boldsymbol{b}}{\lVert \boldsymbol{a} \rVert \, \lVert \boldsymbol{b} \rVert}
$$


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
lda = LinearDiscriminantAnalysis().fit(X_tr, y_tr)
a, b = m.coef_, lda.coef_[0]
cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(round(cos, 4))


コサイン類似度は1.0で、2つの係数の向きは同じです。大きさは違いますが、どの列にどれだけの重みを置くかの比は一致しています。

線形判別分析は、心臓病の人と心臓病でない人の分布の形から、2つのグループをいちばんよく分ける向きを探す方法です。0/1の数を当てる線形回帰とは考え方がまったく違うのに、同じ向きにたどり着きます。違うのは、どこで切るかと、確率を出せるかどうかです。

試すなら、`lda.predict_proba(X_te)`で、線形判別分析が出す確率を見てみましょう。0〜1に収まっているでしょうか。
